# Hugging Face Fundamentals — Lesson 7: Loading Pretrained Models

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `07_Loading_Pretrained_Model.py` (same content, runnable without Jupyter).

**Task ID:** HF-007  |  **Folder:** `07_Loading_Pretrained_Model`


## One API, many sources

`from_pretrained` is more flexible than it looks. It accepts:

- a **Hub id** — `"distilbert/...-sst-2-english"` (what we used so far),
- a **local folder** — downloaded once, then used offline,
- a **revision** — a specific commit, branch or tag,

This lesson exercises all of them.

**Step 1 — the usual Hub load** (cached after the first time):


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
print("loaded from Hub:", model.config.model_type, model.config.num_labels, "labels")


## Step 2 — download once, load from a folder

`snapshot_download` copies a model into a local folder. Loading from that folder uses **no network** — perfect for offline machines and deployments:


In [ ]:
import tempfile
from huggingface_hub import snapshot_download

with tempfile.TemporaryDirectory() as tmp:
    local = snapshot_download(repo_id=model_id, local_dir=tmp)
    local_model = AutoModelForSequenceClassification.from_pretrained(local)
    print("loaded from folder:", type(local_model).__name__)


## Step 3 — save and reload (fine-tuning's finale)

The `save_pretrained` round trip is exactly what HF-208 ends with: a fine-tuned model is saved as a folder and reloaded like any Hub model:


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    model.save_pretrained(tmp)
    tokenizer.save_pretrained(tmp)
    again = AutoModelForSequenceClassification.from_pretrained(tmp)
    print("reloaded from disk, same API, same weights")


## Step 4 — revisions pin your version

Models on the Hub are snapshots. Pin them so your code always gets the same weights:


In [ ]:
pinned = AutoModelForSequenceClassification.from_pretrained(
    model_id, revision="main"   # or a tag / commit hash
)
print("pinned to 'main' — this exact snapshot is reproducible")


## Step 5 — offline mode

With `HF_HUB_OFFLINE=1` nothing touches the network — cached models just work:


In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
try:
    m = AutoModelForSequenceClassification.from_pretrained(model_id)
    print("loaded offline from cache:", type(m).__name__)
finally:
    os.environ.pop("HF_HUB_OFFLINE", None)


## Try it yourself

1. Download any small model with `snapshot_download` and load it.
2. Pin a specific commit hash — then compare some weight values.
3. Turn on `HF_HUB_OFFLINE` and try loading an *uncached* model — error expected!

## Common pitfalls

- **Offline + uncached model = error** — offline only works for cached files.
- **Wrong revision name = 404** — check the Hub's branches/tags.
- **Snapshots can be heavy** — use `ignore_patterns` to skip formats you don't need (`.bin`, `.h5`, `onnx/`).

## Summary

- `from_pretrained`: Hub id, local folder, revision — one API.
- `save_pretrained` + `from_pretrained(folder)` = the share/fine-tune loop.
- `HF_HUB_OFFLINE=1` for fully offline work.

**Next lesson:** HF-008 — Model Cache.  |  Extra reading: `../resources/reference_links.md`
